# House Stage 1 regression

Predictors, target, estimator, years, and holdout years: `src/house/regression/config.py`. Columns: `src/house/regression/schema.py`.
Build the predictors and targets first: `python data-processing/house_features.py` (`modeling.csv`) and `python data-processing/house_targets.py` (`targets.csv`).
Both are loaded whole; `run_experiment` joins the config's target onto the predictors, checks that the keys match, and only then selects years and fits.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../src").resolve()))
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py

In [ ]:
predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
hr.validate_schema(predictors).issues

/var/folders/ln/mpvh0mhs1vg1qh10s96jznxc0000gn/T/ipykernel_73446/945808861.py:1: UserWarning: 
(report.issues lists every affected key)
  predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
/var/folders/ln/mpvh0mhs1vg1qh10s96jznxc0000gn/T/ipykernel_73446/945808861.py:2: UserWarning: 
(report.issues lists every affected key)
  targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)


,column,issue,severity,count,keys
0,gen_margin_raw,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
1,gb_cutoff_date,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
2,gb_date,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
3,gb_margin_raw,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
4,gb_margin_2p_norm,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
5,gb_dem_pct,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
6,gb_gop_pct,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
7,gb_und_pct,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
8,gb_dem_hi,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."
9,gb_dem_lo,null,warning,435,"[(2026, AK, 0), (2026, AL, 1), (2026, AL, 2), ..."


In [ ]:
hr.validate_schema(targets, schema=hr.TARGETS_SCHEMA).issues

,column,issue,severity,count,keys
0,dem_votes,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."
1,gop_votes,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."
2,3p_votes,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."
3,total_votes,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."
4,source_totalvotes,invalid_value,warning,3,"[(2020, FL, 25), (2024, FL, 20), (2024, OK, 3)]"
5,dem_pct,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."
6,gop_pct,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."
7,3p_pct,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."
8,margin_votes,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."
9,margin_raw,null,warning,31,"[(2006, FL, 2), (2006, FL, 3), (2006, FL, 19),..."


In [ ]:
config = hr.MODEL_CONFIGS["classic_ols"]  # or "classic_ols_holdout" (2024), "classic_ols_2026" (forecast)
print(hr.describe_config(config))

model_type    ols
model_params  {'add_constant': True}
feature_set   classic
target        adj_margin_2p_norm
train_years   2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020
eval_years    2022
features      basic_f: [gen_margin_raw, partisan_lean, inc_signed]
use_holdout   False (HOLDOUT_YEARS: [2024])


In [ ]:
joined = hr.join_target(predictors, targets, config["target"])  # what run_experiment selects years from
joined.groupby("year")[config["target"]].count()

year
2006    380
2008    379
2010    406
2012    390
2014    358
2016    372
2018    394
2020    408
2022    403
2024    398
2026      0
Name: adj_margin_2p_norm, dtype: int64

In [ ]:
predictions, model = hr.run_experiment(predictors, targets, config)
model.results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     adj_margin_2p_norm   R-squared:                       0.907
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                 1.004e+04
Date:                Thu, 29 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:11:25   Log-Likelihood:                -11628.
No. Observations:                3087   AIC:                         2.326e+04
Df Residuals:                    3083   BIC:                         2.329e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.4530      0.230     -1.968      0.049      -0.904      -0.002
gen_margin_raw     0.8240      0.031     26.971      0.000       0.764       0.884
partisan_lean      0.8199      0.009     91.227      0.000       0.802       0.837
inc_signed        11.6365      0.281     41.443      0.000      11.086      12.187
==============================================================================
Omnibus:                      224.632   Durbin-Watson:                   1.606
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              913.640
Skew:                           0.244   Prob(JB):                    4.03e-199
Kurtosis:                       5.620   Cond. No.                         43.6
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
hr.score(predictions)  # once every evaluation row has an actual outcome

,year,n,mae,rmse,bias
0,2022,403,6.674291,9.313316,0.018676
1,all,403,6.674291,9.313316,0.018676
